# H-Neurons small Qwen experiment

This notebook is configured for Colab/Kaggle with a single T4 GPU. It uses a Hugging Face model ID directly, so you do not need to download the model manually first.

In [1]:
import os
import os
import pathlib
import subprocess

In [2]:
os.chdir('/kaggle/working')

# Remove existing directory if it exists to force a re-clone
if pathlib.Path('H-Neuron-Implementation').exists():
    print('Removing existing H-Neuron-Implementation directory...')
    subprocess.run(['rm', '-rf', 'H-Neuron-Implementation'], check=True)

# Clone the repository
subprocess.run(['git', 'clone', '--depth=1', '-b', 'dev', '--single-branch', 'https://github.com/CallmeAndree/H-Neuron-Implementation.git'], check=True)



Cloning into 'H-Neuron-Implementation'...


CompletedProcess(args=['git', 'clone', '--depth=1', '-b', 'dev', '--single-branch', 'https://github.com/CallmeAndree/H-Neuron-Implementation.git'], returncode=0)

In [3]:
os.chdir('/kaggle/working/H-Neuron-Implementation')

In [4]:
import subprocess
import sys
# Colab/Kaggle setup. Restart the runtime if vLLM or torch dependencies require it.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 261.0/261.0 MB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 79.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.9/360.9 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.7/13.7 MB 89.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.0/161.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.4 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cud

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], returncode=0)

In [5]:
# Fresh Qwen2.5-7B-Instruct rerun. In fp16 the weights are ~14 GB, so a single T4 (16 GB) will OOM.
# We use a pre-quantized AWQ checkpoint (4-bit) so the model fits comfortably on one T4 (~5-6 GB)
# and vLLM can serve it with AWQ kernels directly (no on-the-fly quantization needed).
# If you prefer to quantize the full-precision checkpoint yourself instead, keep
# MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct' and pass --quantization bitsandbytes below.
MODEL_ID = 'Qwen/Qwen2.5-7B-Instruct-AWQ'
QUANTIZATION = 'awq'  # one of: none, awq, gptq, bitsandbytes

# Set your OpenAI-compatible key only if you run extract_answer_tokens.py.
# In Colab: from google.colab import userdata; OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
# In Kaggle: use Add-ons > Secrets, then read it with kaggle_secrets.
# OPENAI_API_KEY = os.environ.get('OPENAI_API_KEY', 'YOUR_OPENAI_API_KEY')
# BASE_URL = os.environ.get('OPENAI_BASE_URL', 'https://api.openai.com/v1')

OUTPUT_DIR = 'data/small_subset_qwen7b'
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('MODEL_ID =', MODEL_ID)

MODEL_ID = Qwen/Qwen2.5-1.5B-Instruct


## Collect Qwen responses

This generates Qwen-specific responses and rule-based correctness labels. For training, use 5 responses per question as requested. Increase `--max_samples` if you do not get enough balanced true/false samples.

In [6]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
api_key = user_secrets.get_secret("api_key_openai")


In [7]:
BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
BASE_URL_2 = "https://api.vietapi.tech/v1"
LLM_MODEL="models/gemini-3.1-flash-lite"
model = "gpt-5.4"

In [8]:
# WARNING: this file was generated with Qwen2.5-1.5B-Instruct. Sample consistency is model-dependent,
# so for a coherent 7B run you must regenerate the equivalent artifact with the 7B model before enabling this cp.
# !cp /kaggle/input/datasets/vkb0205/h-neuron/Data/small_subset/train_small_qwen.jsonl /kaggle/working

In [9]:
!python h_neuron_scripts/collect_responses.py \
    --model_path {MODEL_ID} \
    --data_path /kaggle/input/datasets/vkb0205/unprocessed-h-neurons/train_split_2.parquet \
    --output_path /kaggle/working/train_small_qwen7b_2.jsonl \
    --sample_num 10 \
    --max_samples 5000 \
    --judge_type llm \
    --api_key {api_key} \
    --base_url {BASE_URL_2} \
    --judge_model {model} \
    --gpu_util 0.7 \
    --quantization {QUANTIZATION} \
    --resume

INFO 06-04 17:27:10 [utils.py:278] non-default args: {'trust_remote_code': True, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.7, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
config.json: 100%|█████████████████████████████| 660/660 [00:00<00:00, 4.63MB/s]
INFO 06-04 17:27:30 [model.py:617] Resolved architecture: Qwen2ForCausalLM
WARNING 06-04 17:27:30 [model.py:2037] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 06-04 17:27:30 [model.py:2090] Casting torch.bfloat16 to torch.float16.
INFO 06-04 17:27:30 [model.py:1752] Using max model len 32768
INFO 06-04 17:27:30 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 06-04 17:27:30 [vllm.py:977] Asynchronous scheduling is enabled.
INFO 06-04 17:27:30 [kernel.py:270] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['nat